# Logistic Regression: Odds Ratios, Cross-Entropy, and Threshold Optimization
Accompanies **Unit 3.3: Logistic Regression from First Principles**.

Logistic regression models conditional log-odds using the sigmoid link function:
$$p(\mathbf{x}) = \sigma(\mathbf{x}^T \mathbf{w}) = \frac{1}{1 + e^{-\mathbf{x}^T \mathbf{w}}}$$
Optimizing binary cross-entropy via Iteratively Reweighted Least Squares (IRLS) yields optimal parameters.

In this notebook, we:
1. Fit logistic regression and interpret coefficients as multiplicative Odds Ratios: $\text{OR}_j = \exp(w_j)$.
2. Calculate Kolmogorov-Smirnov (KS) statistic to determine the optimal probability classification threshold.
3. Evaluate precision, recall, and confusion matrix trade-offs.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report

data_path = os.path.abspath('../../data/bd_train.csv')
df = pd.read_csv(data_path)

numeric_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
X = df[numeric_cols].fillna(df[numeric_cols].median())
y = (df['y'] == 'yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

### Interpreting Multiplicative Odds Ratios
Exponentiating the standardized weights yields the Odds Ratio:
$$\text{OR}_j = \exp(w_j)$$
An OR of $1.50$ implies that a $1.0$ standard deviation increase in that feature multiplies the odds of term deposit subscription by $1.50$ ($50\%$ increase).

In [ ]:
logit = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
logit.fit(X_train_s, y_train)

odds_ratios = pd.DataFrame({
    'Feature': numeric_cols,
    'Standardized Weight': logit.coef_[0],
    'Odds Ratio (exp(w))': np.exp(logit.coef_[0])
}).sort_values(by='Odds Ratio (exp(w))', ascending=False)

print(odds_ratios.to_string(index=False))

### Optimizing the Classification Cutoff via Kolmogorov-Smirnov (KS)
In imbalanced datasets (e.g. only $11\%$ positive subscriptions), the naive $0.50$ cutoff classifies nearly everyone as class $0$.
The KS statistic identifies the threshold that maximizes separation between the cumulative distributions of responders and non-responders:
$$\text{KS} = \max_t |\text{TPR}(t) - \text{FPR}(t)|$$

In [ ]:
y_probs = logit.predict_proba(X_test_s)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_probs)

ks_values = tpr - fpr
optimal_idx = np.argmax(ks_values)
optimal_threshold = thresholds[optimal_idx]

print(f"Maximum KS Statistic:  {ks_values[optimal_idx]:.4f}")
print(f"Optimal Probability Cutoff: {optimal_threshold:.4f} (vs naive 0.50)")

### Confusion Matrix at the Optimal KS Threshold

In [ ]:
y_pred_ks = (y_probs >= optimal_threshold).astype(int)

cm = confusion_matrix(y_test, y_pred_ks)
print("Confusion Matrix at Optimal Cutoff:")
print(pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Pred 0', 'Pred 1']))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ks, target_names=['No Subscription', 'Subscription']))